# Lecture 5 · Optimizers from scratch

Implement SGD, Momentum, Nesterov, RMSProp, Adam, and AdamW in pure NumPy, then race them on a 2D ill-conditioned quadratic.

**Goal** — build up intuition for how each new ingredient (momentum, adaptive LR, bias correction, decoupled decay) changes the trajectory.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1 · The loss surface

An ill-conditioned quadratic with condition number κ = 10 · long narrow ravine along x₁.

In [ ]:
def loss(theta):
    # f(x1, x2) = 10 · x1^2 + x2^2
    return 10 * theta[0]**2 + theta[1]**2

def grad(theta):
    return np.array([20 * theta[0], 2 * theta[1]])

# Contour plot
x1, x2 = np.meshgrid(np.linspace(-5, 5, 200), np.linspace(-5, 5, 200))
Z = 10 * x1**2 + x2**2

fig, ax = plt.subplots(figsize=(6, 5))
ax.contour(x1, x2, Z, levels=np.logspace(-1, 2.5, 15), cmap='Greys')
ax.set_xlabel('θ₁ (steep)'); ax.set_ylabel('θ₂ (gentle)')
ax.set_title('ill-conditioned quadratic · κ = 10')
plt.show()

## 2 · Plain SGD

The baseline. Expect zig-zags along the steep axis.

In [ ]:
def sgd(theta, lr=0.05, steps=200):
    traj = [theta.copy()]
    for _ in range(steps):
        theta = theta - lr * grad(theta)
        traj.append(theta.copy())
    return np.array(traj)

theta0 = np.array([-3.0, 4.0])
traj_sgd = sgd(theta0, lr=0.05, steps=80)
print(f'SGD · final loss = {loss(traj_sgd[-1]):.4f}')

## 3 · SGD with momentum

Exponential moving average of gradients. Damps zig-zag along the steep axis, amplifies progress along the gentle axis.

In [ ]:
def momentum(theta, lr=0.05, beta=0.9, steps=200):
    v = np.zeros_like(theta)
    traj = [theta.copy()]
    for _ in range(steps):
        g = grad(theta)
        v = beta * v + (1 - beta) * g
        theta = theta - lr * v
        traj.append(theta.copy())
    return np.array(traj)

traj_mom = momentum(theta0, lr=0.05, steps=80)
print(f'Momentum · final loss = {loss(traj_mom[-1]):.4f}')

## 4 · Nesterov accelerated gradient

Evaluate the gradient at the *lookahead* point θ − η·β·v.

In [ ]:
def nesterov(theta, lr=0.05, beta=0.9, steps=200):
    v = np.zeros_like(theta)
    traj = [theta.copy()]
    for _ in range(steps):
        lookahead = theta - lr * beta * v
        g = grad(lookahead)
        v = beta * v + (1 - beta) * g
        theta = theta - lr * v
        traj.append(theta.copy())
    return np.array(traj)

traj_nest = nesterov(theta0, lr=0.05, steps=80)
print(f'Nesterov · final loss = {loss(traj_nest[-1]):.4f}')

## 5 · Adam · momentum + RMSProp + bias correction

In [ ]:
def adam(theta, lr=0.1, beta1=0.9, beta2=0.999, eps=1e-8, steps=200):
    m = np.zeros_like(theta)
    v = np.zeros_like(theta)
    traj = [theta.copy()]
    for t in range(1, steps + 1):
        g = grad(theta)
        m = beta1 * m + (1 - beta1) * g
        v = beta2 * v + (1 - beta2) * g**2
        m_hat = m / (1 - beta1**t)
        v_hat = v / (1 - beta2**t)
        theta = theta - lr * m_hat / (np.sqrt(v_hat) + eps)
        traj.append(theta.copy())
    return np.array(traj)

traj_adam = adam(theta0, lr=0.3, steps=80)
print(f'Adam · final loss = {loss(traj_adam[-1]):.4f}')

## 6 · Compare trajectories

Plot all optimizers on the same contour. Observe who zig-zags and who cuts through.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharex=True, sharey=True)
trajs = [('SGD', traj_sgd, '#a8324b'),
         ('Momentum', traj_mom, '#d97757'),
         ('Nesterov', traj_nest, '#7b9e89'),
         ('Adam', traj_adam, '#4a6670')]

for ax, (name, traj, color) in zip(axes, trajs):
    ax.contour(x1, x2, Z, levels=np.logspace(-1, 2.5, 15), cmap='Greys', alpha=0.5)
    ax.plot(traj[:, 0], traj[:, 1], color=color, linewidth=2, marker='o', markersize=3)
    ax.scatter(0, 0, marker='*', c='#7b9e89', s=200, edgecolor='black', zorder=10, label='min')
    ax.set_title(f'{name}\nfinal loss = {loss(traj[-1]):.3f}')
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
plt.tight_layout()
plt.show()

## 7 · Loss curves · who converges fastest

On this ill-conditioned quadratic, momentum and Adam crush SGD.

In [ ]:
plt.figure(figsize=(10, 5))
for name, traj, color in trajs:
    losses = [loss(t) for t in traj]
    plt.semilogy(losses, label=name, color=color, linewidth=2)
plt.xlabel('step'); plt.ylabel('loss (log)')
plt.title('convergence on ill-conditioned quadratic')
plt.legend(); plt.grid(True, alpha=0.3)
plt.show()

## 8 · Bias correction · the numeric evidence

Verify the bias-correction factor by running Adam **without** the correction and comparing the first few steps.

In [ ]:
# Constant gradient of 1 — what should m_t track?
beta1 = 0.9
m_uncorr = 0.0
print(f'{"step":>6} {"m_t":>10} {"m_hat":>10}')
for t in range(1, 11):
    m_uncorr = beta1 * m_uncorr + (1 - beta1) * 1.0
    m_corr = m_uncorr / (1 - beta1**t)
    print(f'{t:>6} {m_uncorr:>10.4f} {m_corr:>10.4f}')

The raw EMA is 10× too small at t=1. After bias correction, each row reads 1.0 — the true gradient. This is why Adam's first few steps are calibrated correctly.

---

**Reflection**
- **SGD** · fine for well-conditioned problems, hopeless for ravines.
- **Momentum** · adds inertia; dampens noise; 2× to 10× faster in practice.
- **Nesterov** · small extra win; one flag in PyTorch.
- **Adam** · per-parameter LR + bias correction. Robust default for everything except pure-vision CNNs.

**Try next** · run SGD with `lr = 0.1` · it will diverge. Ill-conditioning + no momentum = overshoot.